# Phase 1: Descriptive Analysis of Sugar Intake, Smoking, and Tooth Decay
**Student:** Mohammad Hossein Amini (403100065)
**Course:** Statistics and Its Applications
**Instructor:** Dr. Kasra Alishahi

## 1. Introduction
This notebook performs the descriptive analysis for Phase 1 of the project. The goal is to investigate the association between total sugar intake, smoking status, and tooth decay in US adults using the NHANES 2021-2023 dataset.

In [42]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set plot style
sns.set_theme(style="whitegrid")
print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Data Loading and Merging
We load four datasets and merge them using the unique identifier `SEQN`.

In [43]:
# Load raw datasets
demo = pd.read_sas("../data/DEMO_L.xpt", format="xport")
dr1tot = pd.read_sas("../data/DR1TOT_L.xpt", format="xport")
ohq = pd.read_sas("../data/OHQ_L.xpt", format="xport")
smq = pd.read_sas("../data/SMQ_L.xpt", format="xport")

# Data Flow: Count before merging
raw_counts = {
    'DEMO': len(demo),
    'DR1TOT': len(dr1tot),
    'OHQ': len(ohq),
    'SMQ': len(smq)
}
print("Raw dataset sizes:", raw_counts)

# Merge datasets
demo_cols = ['SEQN', 'RIAGENDR', 'RIDAGEYR', 'WTMEC2YR']
dr1tot_cols = ['SEQN', 'DR1TSUGR']
ohq_cols = ['SEQN', 'OHQ660']
smq_cols = ['SEQN', 'SMQ020']

merged1 = demo[demo_cols].merge(dr1tot[dr1tot_cols], on='SEQN', how='inner')
merged2 = merged1.merge(ohq[ohq_cols], on='SEQN', how='inner')
merged3 = merged2.merge(smq[smq_cols], on='SEQN', how='inner')

print(f"After merging: {len(merged3)} rows")

Raw dataset sizes: {'DEMO': 11933, 'DR1TOT': 8860, 'OHQ': 11744, 'SMQ': 9015}
After merging: 7199 rows


## 3. Data Cleaning
We handle missing data, reclassify the tooth decay variable (1,2,3,4 = Yes; 5 = No), and filter for adults (age >= 18).

In [44]:
# Replace special codes (7, 9) with NaN
merged3 = merged3.replace({7: pd.NA, 9: pd.NA})

# Reclassify OHQ660: 1,2,3,4 = Yes (1); 5 = No (0)
merged3['OHQ660'] = merged3['OHQ660'].replace({1.0: 1, 2.0: 1, 3.0: 1, 4.0: 1, 5.0: 0})

# Drop missing values in primary variables
merged3 = merged3.dropna(subset=['DR1TSUGR', 'OHQ660', 'SMQ020'])

# Filter adults (age >= 18)
df = merged3[merged3['RIDAGEYR'] >= 18].copy()

# Convert gender to labels
df['RIAGENDR'] = df['RIAGENDR'].replace({1.0: 'Male', 2.0: 'Female'})

# Map SMQ020 for better labels (1 = Smoker, 2 = Non-Smoker)
df['Smoking_Label'] = df['SMQ020'].map({1.0: 'Smoker', 2.0: 'Non-Smoker'})

print(f"Final analytical sample size: {len(df)}")
print(df.head())

Final analytical sample size: 4788
       SEQN RIAGENDR  RIDAGEYR      WTMEC2YR  DR1TSUGR OHQ660 SMQ020  \
0  130378.0     Male      43.0  54374.463898     43.71      0    1.0   
1  130379.0     Male      66.0  34084.721548    113.58      0    1.0   
2  130380.0   Female      44.0  81196.277992    114.66      1    2.0   
3  130386.0     Male      34.0  39988.452940     37.89      0    1.0   
4  130387.0   Female      68.0  20776.254850    227.99      0    2.0   

  Smoking_Label  
0        Smoker  
1        Smoker  
2    Non-Smoker  
3        Smoker  
4    Non-Smoker  


## 4. Missing Data Audit
We calculate the percentage of missing data before cleaning.

In [45]:
# Re-create raw merged for audit
raw_merged = demo[demo_cols].merge(
    dr1tot[dr1tot_cols], on='SEQN', how='inner'
).merge(
    ohq[ohq_cols], on='SEQN', how='inner'
).merge(
    smq[smq_cols], on='SEQN', how='inner'
)

missing_pct = raw_merged.isnull().sum() / len(raw_merged) * 100
print("Missing Data Percentage (Before Cleaning):")
print(missing_pct.round(2))

Missing Data Percentage (Before Cleaning):
SEQN         0.00
RIAGENDR     0.00
RIDAGEYR     0.00
WTMEC2YR     0.00
DR1TSUGR    22.82
OHQ660      15.78
SMQ020      12.00
dtype: float64


## 5. Descriptive Statistics
We calculate mean, median, standard deviation, variance, IQR, skewness, and kurtosis.

In [46]:
print("=== Descriptive Statistics ===")
for col in ['DR1TSUGR', 'RIDAGEYR']:
    desc = df[col].describe()
    iqr = desc['75%'] - desc['25%']
    variance = df[col].var()
    skewness = df[col].skew()
    kurt = df[col].kurtosis()
    print(f"\n{col}:")
    print(f"  Mean: {desc['mean']:.2f} | Median: {desc['50%']:.2f} | SD: {desc['std']:.2f}")
    print(f"  Variance: {variance:.2f} | IQR: {iqr:.2f}")
    print(f"  Skewness: {skewness:.2f} | Kurtosis: {kurt:.2f}")

print(f"\nMode of Tooth Decay: {df['OHQ660'].mode()[0]}")
print(f"Mode of Smoking: {df['SMQ020'].mode()[0]}")
print(f"Mode of Gender: {df['RIAGENDR'].mode()[0]}")

=== Descriptive Statistics ===

DR1TSUGR:
  Mean: 95.47 | Median: 81.12 | SD: 68.13
  Variance: 4641.38 | IQR: 72.59
  Skewness: 2.31 | Kurtosis: 12.74

RIDAGEYR:
  Mean: 55.05 | Median: 59.00 | SD: 16.84
  Variance: 283.53 | IQR: 27.00
  Skewness: -0.38 | Kurtosis: -1.00

Mode of Tooth Decay: 0
Mode of Smoking: 2.0
Mode of Gender: Female


## 6. Frequency Tables

In [47]:
print("=== Frequency Tables ===")
print("\nGender:\n", df['RIAGENDR'].value_counts(normalize=True).round(4) * 100)
print("\nTooth Decay:\n", df['OHQ660'].value_counts(normalize=True).rename({0.0: 'No', 1.0: 'Yes'}).round(4) * 100)
print("\nSmoking:\n", df['SMQ020'].value_counts(normalize=True).rename({2.0: 'No', 1.0: 'Yes'}).round(4) * 100)

=== Frequency Tables ===

Gender:
 RIAGENDR
Female    55.93
Male      44.07
Name: proportion, dtype: float64

Tooth Decay:
 OHQ660
No     62.82
Yes    37.18
Name: proportion, dtype: float64

Smoking:
 SMQ020
No     58.67
Yes    41.33
Name: proportion, dtype: float64


## 7. Comparison by Gender and Weighted Analysis

In [48]:
print("\n=== Sugar Intake by Gender ===")
print(df.groupby('RIAGENDR')['DR1TSUGR'].agg(['mean', 'median', 'std']).round(2))

print("\n=== Weighted Proportions ===")
w_tooth_decay = (df['OHQ660'] * df['WTMEC2YR']).sum() / df['WTMEC2YR'].sum() * 100
w_smoking = (df['SMQ020'] * df['WTMEC2YR']).sum() / df['WTMEC2YR'].sum() * 100
print(f"Weighted % of Tooth Decay: {w_tooth_decay:.2f}%")
print(f"Weighted % of Smokers: {w_smoking:.2f}%")

raw = df['DR1TSUGR'].mean()
weighted = (df['DR1TSUGR'] * df['WTMEC2YR']).sum() / df['WTMEC2YR'].sum()
print(f"Raw Sugar Mean: {raw:.2f} | Weighted Sugar Mean: {weighted:.2f}")


=== Sugar Intake by Gender ===
            mean  median    std
RIAGENDR                       
Female     85.75   74.98  56.90
Male      107.79   91.99  78.44

=== Weighted Proportions ===
Weighted % of Tooth Decay: 34.65%
Weighted % of Smokers: 162.02%
Raw Sugar Mean: 95.47 | Weighted Sugar Mean: 95.58


## 8. Visualizations
We generate 14 meaningful plots to visualize the data.

In [41]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load Clean Data
df = pd.read_csv("../data/clean_data.csv")

# Map SMQ020 to labels for better visualization (1.0 = Smoker, 2.0 = Non-Smoker)
df['Smoking_Label'] = df['SMQ020'].map({1.0: 'Smoker', 2.0: 'Non-Smoker'})

# Set style
sns.set_theme(style="whitegrid")

# ==========================================
# 1. Histogram of Sugar Intake
# ==========================================
plt.figure(figsize=(8, 5))
sns.histplot(df['DR1TSUGR'], kde=True, color='royalblue', bins=50)
plt.axvline(df['DR1TSUGR'].mean(), color='red', linestyle='--', label=f"Mean: {df['DR1TSUGR'].mean():.1f}")
plt.axvline(df['DR1TSUGR'].median(), color='green', linestyle='-', label=f"Median: {df['DR1TSUGR'].median():.1f}")
plt.title('Distribution of Total Sugar Intake')
plt.xlabel('Sugar (grams)')
plt.ylabel('Frequency')
plt.legend()
plt.savefig('../figures/hist_sugar.png', dpi=300)
plt.close()

# ==========================================
# 2. Boxplot of Sugar by Tooth Decay
# ==========================================
plt.figure(figsize=(8, 5))
sns.boxplot(x='OHQ660', y='DR1TSUGR', data=df, hue='OHQ660', palette='Set2', legend=False)
plt.xticks([0, 1], ['No Decay', 'Tooth Decay'])
plt.title('Sugar Intake by Tooth Decay Status')
plt.xlabel('Tooth Decay')
plt.ylabel('Sugar (grams)')
plt.savefig('../figures/box_sugar_tooth.png', dpi=300)
plt.close()

# ==========================================
# 3. Boxplot of Sugar by Smoking
# ==========================================
plt.figure(figsize=(8, 5))
sns.boxplot(x='Smoking_Label', y='DR1TSUGR', data=df, hue='Smoking_Label', palette='Set1', legend=False)
plt.title('Sugar Intake by Smoking Status')
plt.xlabel('Smoking Status')
plt.ylabel('Sugar (grams)')
plt.savefig('../figures/box_sugar_smoking.png', dpi=300)
plt.close()

# ==========================================
# 4. Scatterplot (Sugar vs Age)
# ==========================================
plt.figure(figsize=(8, 5))
sns.scatterplot(x='RIDAGEYR', y='DR1TSUGR', data=df, alpha=0.3, color='teal')
plt.title('Sugar Intake vs Age')
plt.xlabel('Age (years)')
plt.ylabel('Sugar (grams)')
plt.savefig('../figures/scatter_sugar_age.png', dpi=300)
plt.close()

# ==========================================
# 5. Countplot of Tooth Decay by Gender
# ==========================================
plt.figure(figsize=(8, 5))
sns.countplot(x='RIAGENDR', hue='OHQ660', data=df, palette='coolwarm')
plt.title('Tooth Decay Frequency by Gender')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.legend(title='Tooth Decay', labels=['No', 'Yes'])
plt.savefig('../figures/bar_tooth_gender.png', dpi=300)
plt.close()

# ==========================================
# 6. Barplot of Mean Sugar by Age Group
# ==========================================
df['AgeGroup'] = pd.cut(df['RIDAGEYR'], bins=[18, 30, 45, 60, 80], labels=['18-30', '31-45', '46-60', '60+'])
plt.figure(figsize=(8, 5))
sns.barplot(x='AgeGroup', y='DR1TSUGR', data=df, hue='AgeGroup', palette='viridis', errorbar=None, legend=False)
plt.title('Mean Sugar Intake by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Mean Sugar (grams)')
plt.savefig('../figures/bar_sugar_agegroup.png', dpi=300)
plt.close()

# ==========================================
# 7. Line Plot: Mean Sugar Intake by Age Group
# ==========================================
plt.figure(figsize=(8, 5))
sns.lineplot(x='AgeGroup', y='DR1TSUGR', data=df, marker='o', color='purple')
plt.title('Trend of Mean Sugar Intake Across Age Groups')
plt.xlabel('Age Group')
plt.ylabel('Mean Sugar (grams)')
plt.savefig('../figures/line_sugar_agegroup.png', dpi=300)
plt.close()

# ==========================================
# 8. Interaction Plot: Sugar by Tooth Decay, Split by Gender
# ==========================================
plt.figure(figsize=(8, 5))
sns.catplot(x='OHQ660', y='DR1TSUGR', hue='RIAGENDR', data=df, kind='bar', palette='Set1', height=5, aspect=1.5)
plt.title('Interaction of Tooth Decay and Gender on Sugar Intake')
plt.xlabel('Tooth Decay (0=No, 1=Yes)')
plt.ylabel('Mean Sugar (grams)')
plt.savefig('../figures/interaction_sugar_tooth_gender.png', dpi=300)
plt.close()

# ==========================================
# 9. Split Boxplot: Sugar by Smoking and Tooth Decay
# ==========================================
plt.figure(figsize=(8, 5))
sns.boxplot(x='Smoking_Label', y='DR1TSUGR', hue='OHQ660', data=df, palette='Set2')
plt.title('Sugar Intake by Smoking and Tooth Decay')
plt.xlabel('Smoking Status')
plt.ylabel('Sugar (grams)')
plt.legend(title='Tooth Decay', labels=['No', 'Yes'])
plt.savefig('../figures/box_sugar_smoking_tooth.png', dpi=300)
plt.close()

# ==========================================
# 10. KDE Plot: Distribution of Sugar Intake by Gender
# ==========================================
plt.figure(figsize=(8, 5))
sns.kdeplot(data=df, x='DR1TSUGR', hue='RIAGENDR', fill=True, palette='coolwarm', alpha=0.5)
plt.title('Density of Sugar Intake by Gender')
plt.xlabel('Sugar (grams)')
plt.ylabel('Density')
plt.savefig('../figures/kde_sugar_gender.png', dpi=300)
plt.close()

# ==========================================
# 11. Pie Charts: Tooth Decay Percentage by Smoking Status
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Non-smokers
non_smokers = df[df['Smoking_Label'] == 'Non-Smoker']['OHQ660'].value_counts(normalize=True)
axes[0].pie(non_smokers, labels=['No Decay', 'Decay'], autopct='%1.1f%%', colors=['lightgreen', 'lightcoral'], startangle=90)
axes[0].set_title('Tooth Decay in Non-Smokers')

# Smokers
smokers = df[df['Smoking_Label'] == 'Smoker']['OHQ660'].value_counts(normalize=True)
axes[1].pie(smokers, labels=['No Decay', 'Decay'], autopct='%1.1f%%', colors=['lightgreen', 'lightcoral'], startangle=90)
axes[1].set_title('Tooth Decay in Smokers')

plt.tight_layout()
plt.savefig('../figures/pie_tooth_smoking.png', dpi=300)
plt.close()

# ==========================================
# 12. Correlation Heatmap (Descriptive)
# ==========================================
plt.figure(figsize=(8, 6))
# Select numerical columns for correlation
corr_cols = ['DR1TSUGR', 'RIDAGEYR', 'OHQ660', 'SMQ020']
corr_matrix = df[corr_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Heatmap (Descriptive)')
plt.tight_layout()
plt.savefig('../figures/correlation_heatmap.png', dpi=300)
plt.close()

import numpy as np
from scipy import stats

# ==========================================
# 13. Additional Descriptive Statistics (Skewness, Kurtosis, Mode)
# ==========================================
print("\n=== Shape Statistics ===")
for col in ['DR1TSUGR', 'RIDAGEYR']:
    skewness = df[col].skew()
    kurt = df[col].kurtosis()
    print(f"{col}: Skewness = {skewness:.2f}, Kurtosis = {kurt:.2f}")

# Mode for categorical/binary variables
print(f"Mode of Tooth Decay (OHQ660): {df['OHQ660'].mode()[0]}")
print(f"Mode of Smoking (SMQ020): {df['SMQ020'].mode()[0]}")
print(f"Mode of Gender (RIAGENDR): {df['RIAGENDR'].mode()[0]}")

# ==========================================
# 14. Weighted Proportions for Binary Variables
# ==========================================
print("\n=== Weighted Proportions ===")
# Weighted proportion of Tooth Decay (Yes = 1)
w_tooth_decay = (df['OHQ660'] * df['WTMEC2YR']).sum() / df['WTMEC2YR'].sum() * 100
print(f"Weighted % of Tooth Decay: {w_tooth_decay:.2f}%")

# Weighted proportion of Smoking (Yes = 1)
w_smoking = (df['SMQ020'] * df['WTMEC2YR']).sum() / df['WTMEC2YR'].sum() * 100
print(f"Weighted % of Smokers: {w_smoking:.2f}%")

# ==========================================
# 15. Additional Plots (QQ-Plot & Age Boxplot)
# ==========================================
# 15.1 QQ-Plot for Sugar
plt.figure(figsize=(6, 5))
stats.probplot(df['DR1TSUGR'], dist="norm", plot=plt)
plt.title('QQ-Plot of Total Sugar Intake')
plt.savefig('../figures/qq_sugar.png', dpi=300)
plt.close()

# 15.2 Boxplot of Age by Tooth Decay
plt.figure(figsize=(8, 5))
sns.boxplot(x='OHQ660', y='RIDAGEYR', data=df, hue='OHQ660', palette='Set2', legend=False)
plt.xticks([0, 1], ['No Decay', 'Tooth Decay'])
plt.title('Age Distribution by Tooth Decay Status')
plt.xlabel('Tooth Decay')
plt.ylabel('Age (years)')
plt.savefig('../figures/box_age_tooth.png', dpi=300)
plt.close()

print("\n✅ All 14 plots and full descriptive statistics have been generated successfully.")


=== Shape Statistics ===
DR1TSUGR: Skewness = 2.31, Kurtosis = 12.74
RIDAGEYR: Skewness = -0.38, Kurtosis = -1.00
Mode of Tooth Decay (OHQ660): 0
Mode of Smoking (SMQ020): 2.0
Mode of Gender (RIAGENDR): Female

=== Weighted Proportions ===
Weighted % of Tooth Decay: 34.65%
Weighted % of Smokers: 162.02%

✅ All 14 plots and full descriptive statistics have been generated successfully.


<Figure size 800x500 with 0 Axes>